# Test Runner Notebook

Tests can be triggered from here manually before pushing (in future this may be possible to make part of a push hook so its automatic but that is out of scope for now).

GitActions triggers the unit tests within the github environment current for each interaction.

GitActions triggers this notebook via a pipeline within the databricks environment. This is only necessary for integration and data tests which need to happen within the databricks environment. (With DI and mocking we could do a certain amount of the integration tests in github environment but it is not necessary to run them there yet, though should still making the tests in this way).

The job passes values for @pytest.mark values 

⚠️ currently the github test runner in ci.yml is not using this file (others are by triggering it through dtabricks) TODO qqqq
so changes need replicating here for github run unit tests /.github/workflows/ci.yml, such as .mark

## Test Env Setup

In [0]:
# MAGIC %pip install pytest
# because doesnt get pytest from toml???
import pytest
import sys
import os

# We need the catalog for our temp storage volume
# we also need it if data tests
# Get catalog from job
dbutils.widgets.text("job.catalog", "dev_catalog") # should default to bundle value if exists
catalog = dbutils.widgets.get("job.catalog")
print(catalog)

# we want to exclude some tests based on environment like freshness in manual run
dbutils.widgets.text("job.pytest_marks", "not dev_skip") # e.g local user environment
default_markers = dbutils.widgets.get("job.pytest_marks")

dbutils.widgets.text("job.schema_prefix", "")
schema_prefix = dbutils.widgets.get("job.schema_prefix") #remember username trailing underscore
print(schema_prefix)

# qqqq we cant get bundle vars from spark unless the bundle is deployed so we need logic so we need a fall back for running locally in personal area

# qqqq these var passings shouldnt be the best way need to see a working example
# So pytest can get it from the environment
os.environ["CATALOG"] = catalog
os.environ["SCHEMA_PREFIX"] = schema_prefix
# This looks 'up' one level from this notebook to find your project root
# using toml now
# repo_root = os.path.abspath('..') # sys.path.append(f"{repo_root}/src")

# Prevent __pycache__ creation
sys.dont_write_bytecode = True

# print(f"Project root set to: {repo_root}")
print("Setup run")

## Unit Test Runner

In [0]:

#%skip
# Run your test file
# This will run every file starting with 'test_' in that folder
args = [
    "unit-tests",
    "-v",                # verbose
    "-s",                # show print output
    "--tb=short"         # short tracebacks
]

# can do it inline but empty values might break it so this is safer
if default_markers:
    args.extend(["-m", default_markers])
unit_exit_code = pytest.main(args)

print(f"Unit test exit code: {unit_exit_code}")

## Integration Test Runner

In [0]:
#%skip
# Run your test file
# This will run every file starting with 'test_' in that folder

# Default marker for this run (you want to exclude sql/manual)
integration_test_marks = "not integration_sql and not manual"

# Combine default + widget (if widget exists)
combined_markers = " and ".join(filter(None, [default_markers, integration_test_marks]))


integration_exit_code = pytest.main([
    "integration-tests", 
    "-m", combined_markers, # exclude sql tests i want them to run seperately below
    "-v",                # verbose
    "-s",                # show print output
    "--tb=short"         # short tracebacks
])

print(f"Integration test exit code: {integration_exit_code}")

### SQL Integration Tests

## Not sure we should do SQL based tests but this is an example
The pyton runs the sql test and then can fail the pipeline (which is what we want it to do as its acting as a testrunner)

Time may be better translating into python that unit testing it

In [0]:
#%skip
# i think we should turn stored procedure into python functions, not test them this way but it depends on practicality
# there is an example of the function equivalent (close enough) in this solution

# Default marker for this run (you want to exclude sql/manual)
sql_integration_test_marks = "not integration_sql and not manual"

# Combine default + widget (if widget exists)
combined_markers = " and ".join(filter(None, [default_markers, sql_integration_test_marks]))

integration_sql_exit_code = pytest.main([
    "integration-tests", 
    "-m", combined_markers, 
    "-v",                # verbose
    "-s",                # show print output
    "--tb=short"         # short tracebacks
])

print(f"Integration test exit code: {integration_sql_exit_code}")

# Data Quality Test Runner

Relying on actual data so is like an integration test and needs to run in databricks

In [0]:
#%skip
# Run your test file
# This will run every file starting with 'test_' in that folder

# Default marker for this run (you want to exclude sql/manual)
data_quality_test_marks = "not integration_sql and not manual"

# Combine default + widget (if widget exists)
combined_markers = " and ".join(filter(None, [default_markers, sql_integration_test_marks]))

data_quality_exit_code = pytest.main([
    "data-quality-tests",
    "-m", combined_markers,
    "-v",                # verbose
    "-s",                # show print output
    "--tb=short"         # short tracebacks
])
print("default_markers:", default_markers)
print("sql_integration_test_marks:", sql_integration_test_marks)
print("combined_markers:", combined_markers)
print(f"data quality test exit code: {data_quality_exit_code}")

# Fail or pass the pipeline


In [0]:
failures = {
    "unit": unit_exit_code,
    "integration": integration_exit_code,
    "integration-sql": integration_sql_exit_code,
    "data-quality": data_quality_exit_code
}

failed = {k: v for k, v in failures.items() if v != 0}

if failed:
    print("❌ Test failures detected:")
    for name, code in failed.items():
        print(f" - {name} tests failed with exit code {code}")

    # Fail the Databricks job
    raise RuntimeError(f"Tests failed: {failed}")

print("✅ All tests passed")

# Out of scope: code coverage
- Want to see direction code is going
- Like TELBlazor would be nice to inforce increase in coverage
  - and html report gitpage is a nice to have to